# Compustat Cleaning And CRSP Merge

This notebook processes `compustat_20-25.csv` and then merges it with monthly CRSP data:
- load and inspect the dataset
- remove duplicate firm-quarter observations
- clean and repair `rdq`
- summarize and handle missing values
- add `LPERMNO` from `ccm_link.csv`
- create accounting ratios using only Compustat columns
- save the cleaned Compustat output
- merge with `crsp_monthly_clean.csv` using `LPERMNO = PERMNO`
- apply the rule that each quarter becomes active in the month of its `rdq`
- save both the full merged dataset and the matched-only modeling dataset
- summarize missing percentages for the matched-only dataset

`BM` is not calculated here because this notebook only uses columns from `compustat_20-25.csv` for accounting feature engineering.

## 1. Setup And File Paths

In [1]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path('.')
COMPUSTAT_PATH = BASE_DIR / 'compustat_20-25.csv'
CCM_PATH = BASE_DIR / 'ccm_link.csv'
CRSP_MONTHLY_PATH = BASE_DIR / 'Stock Data' / 'crsp_monthly_clean.csv'
OUTPUT_PATH = BASE_DIR / 'compustat_cleaned.csv'
MISSING_PATH = BASE_DIR / 'compustat_missing_pct.csv'
MERGED_OUTPUT_PATH = BASE_DIR / 'compustat_crsp_merged.csv'
MATCHED_ONLY_OUTPUT_PATH = BASE_DIR / 'compustat_crsp_merged_matched_only.csv'
MATCHED_ONLY_MISSING_PATH = BASE_DIR / 'compustat_crsp_merged_matched_only_missing_pct.csv'

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

## 2. Load And Inspect Compustat Data

In [2]:
compustat = pd.read_csv(COMPUSTAT_PATH)
compustat['datadate'] = pd.to_datetime(compustat['datadate'], errors='coerce')
compustat['rdq'] = pd.to_datetime(compustat['rdq'], errors='coerce')
compustat['gvkey'] = compustat['gvkey'].astype(str).str.zfill(6)

print('Loaded Compustat data:', compustat.shape)
compustat.head()

Loaded Compustat data: (344417, 31)


,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,cusip,naics,sic,fqtr,fyearq,rdq,actq,atq,ceqq,cheq,cogsq,dlcq,dlttq,lctq,ltq,niq,oibdpq,saleq,xsgaq,capxy,oancfy,prccq,prclq
0,A,USD,STD,INDL,C,ABM,2019-01-31,001410,000957100,561720.0,7340,1,2019,2019-03-06,1231.100,3686.400,1461.100,30.600,1434.400,42.1,945.700,798.000,2225.300,13.000,60.900,1607.900,112.600,11.600,-39.300,34.19,25.64
1,I,USD,STD,INDL,C,LGTY,2019-01-31,001562,029683109,513210.0,7372,3,2018,2019-02-20,112.638,160.821,114.356,83.164,11.101,0.0,0.000,42.341,46.465,2.301,4.090,27.003,11.812,1.014,13.608,11.06,8.99
2,A,USD,STD,INDL,C,AXR,2019-01-31,001618,032159105,237210.0,6552,3,2018,2019-03-13,NaN,104.510,87.971,14.233,2.093,0.0,2.554,NaN,16.539,-0.032,-0.634,2.381,0.922,0.062,-0.524,6.22,5.47
3,A,USD,STD,INDL,C,ADI,2019-01-31,001632,032654105,334413.0,3674,1,2019,2019-02-20,1985.180,21828.278,11585.437,605.864,408.184,0.0,6234.517,849.955,10242.841,355.006,678.193,1541.101,454.724,90.993,371.767,98.86,80.08
4,A,USD,STD,INDL,C,AMAT,2019-01-31,001704,038222105,333242.0,3559,1,2019,2019-02-14,10285.000,18922.000,8209.000,3712.000,2005.000,0.0,5310.000,3776.000,10713.000,771.000,999.000,3753.000,749.000,133.000,834.000,39.08,28.79


## 3. Deduplicate Firm-Quarter Records

In [3]:
# Keep the most complete row for each gvkey-datadate pair.
compustat['_non_missing_count'] = compustat.notna().sum(axis=1)
compustat = compustat.sort_values(
    ['gvkey', 'datadate', '_non_missing_count'],
    ascending=[True, True, False]
)

duplicates_removed = len(compustat) - len(compustat.drop_duplicates(subset=['gvkey', 'datadate'], keep='first'))
compustat = compustat.drop_duplicates(subset=['gvkey', 'datadate'], keep='first').drop(columns='_non_missing_count')

print('Removed duplicate rows:', duplicates_removed)
print('Shape after deduplication:', compustat.shape)

Removed duplicate rows: 140
Shape after deduplication: (344277, 31)


## 4. Clean And Repair RDQ

In [4]:
# Rule 1: fill missing rdq with datadate + 30 days.
rdq_missing_mask = compustat['rdq'].isna()
compustat.loc[rdq_missing_mask, 'rdq'] = compustat.loc[rdq_missing_mask, 'datadate'] + pd.Timedelta(days=30)

# Rule 2: if the same company uses the same rdq across multiple quarters,
# treat those repeated values as implausible and reset them to datadate + 30 days.
rdq_repeated_mask = compustat.duplicated(subset=['gvkey', 'rdq'], keep=False)
compustat.loc[rdq_repeated_mask, 'rdq'] = compustat.loc[rdq_repeated_mask, 'datadate'] + pd.Timedelta(days=30)

# Rule 3: if collisions still remain after the replacement, add a small within-group
# day offset so each company-quarter ends up with a unique rdq.
rdq_collision_order = compustat.groupby(['gvkey', 'rdq']).cumcount()
compustat['rdq'] = compustat['rdq'] + pd.to_timedelta(rdq_collision_order, unit='D')

rdq_filled_count = int(rdq_missing_mask.sum())
rdq_repaired_count = int(rdq_repeated_mask.sum())
remaining_repeated_rdq = int(compustat.duplicated(subset=['gvkey', 'rdq']).sum())

print('Filled missing rdq rows:', rdq_filled_count)
print('Repaired repeated rdq rows within company:', rdq_repaired_count)
print('Remaining duplicated gvkey-rdq pairs:', remaining_repeated_rdq)
compustat[['gvkey', 'datadate', 'rdq']].head(10)

Filled missing rdq rows: 137090
Repaired repeated rdq rows within company: 3339
Remaining duplicated gvkey-rdq pairs: 0


,gvkey,datadate,rdq
641,001004,2019-02-28,2019-03-19
12304,001004,2019-05-31,2019-07-10
23934,001004,2019-08-31,2019-09-25
35481,001004,2019-11-30,2019-12-19
46876,001004,2020-02-29,2020-03-24
58666,001004,2020-05-31,2020-07-21
70435,001004,2020-08-31,2020-09-24
82164,001004,2020-11-30,2020-12-17
93858,001004,2021-02-28,2021-03-23
105971,001004,2021-05-31,2021-07-20


## 5. Explain RDQ Repair Rule

This notebook uses a simple, reproducible rule to repair `rdq`:
- if `rdq` is missing, replace it with `datadate + 30 days`
- if the same company has the same `rdq` repeated across multiple quarters, replace those repeated values with `datadate + 30 days`
- if repeated values still remain after that replacement, add a small within-group day offset to keep `rdq` unique within each company

This is a pragmatic data-cleaning choice for the project. It does not claim to recover the true filing date. Instead, it prevents obviously problematic `rdq` values from distorting later time alignment.

## 6. Analyze Missing Values And Remove Selected Columns

In [5]:
missing_pct = (
    (compustat.isna().mean() * 100)
    .round(2)
    .rename('missing_pct')
    .reset_index()
    .rename(columns={'index': 'column'})
    .sort_values('missing_pct', ascending=False)
)

missing_pct.to_csv(MISSING_PATH, index=False)

# Only drop lower-priority, high-missing columns if they actually exist.
candidate_drop_columns = ['actq', 'lctq', 'xsgaq', 'oibdpq', 'capxy', 'oancfy']
drop_columns = [col for col in candidate_drop_columns if col in compustat.columns]
compustat = compustat.drop(columns=drop_columns)

print('Dropped columns:', drop_columns)
print('Shape after selective missing-value column removal:', compustat.shape)
missing_pct.head(15)

Dropped columns: ['actq', 'lctq', 'xsgaq', 'oibdpq', 'capxy', 'oancfy']
Shape after selective missing-value column removal: (344277, 25)


,column,missing_pct
14,actq,49.54
21,lctq,49.52
26,xsgaq,48.61
19,dlcq,43.26
24,oibdpq,42.15
27,capxy,41.64
28,oancfy,41.18
20,dlttq,39.90
16,ceqq,39.54
17,cheq,39.51


## 7. Explain Missing-Value Decisions

Columns above 40% missing are not dropped automatically. We only remove columns that satisfy all three conditions:
- they have very high missingness,
- they are not required for the current data engineering,
- and they are lower-priority features for an initial excess-return prediction baseline.

The notebook also checks whether those candidate columns exist before dropping them, so it stays robust when your updated Compustat file adds or removes variables.

## 8. Match LPERMNO From CCM Link Table

In [6]:
ccm = pd.read_csv(CCM_PATH, dtype={'gvkey': str})
ccm['gvkey'] = ccm['gvkey'].astype(str).str.zfill(6)
ccm['LINKDT'] = pd.to_datetime(ccm['LINKDT'], errors='coerce')
ccm['LINKENDDT'] = pd.to_datetime(ccm['LINKENDDT'].replace('E', pd.NA), errors='coerce')
ccm['LINKENDDT'] = ccm['LINKENDDT'].fillna(pd.Timestamp('today').normalize())

ccm = ccm[ccm['LINKTYPE'].isin(['LC', 'LU'])].copy()
ccm = ccm[ccm['LINKPRIM'].isin(['P', 'C'])].copy()
ccm = ccm.sort_values(['gvkey', 'LINKDT', 'LINKENDDT'])

compustat_lpermno = compustat.merge(ccm[['gvkey', 'LPERMNO', 'LINKDT', 'LINKENDDT']], on='gvkey', how='left')
compustat_lpermno = compustat_lpermno[
    compustat_lpermno['LINKDT'].isna()
    | (
        (compustat_lpermno['datadate'] >= compustat_lpermno['LINKDT'])
        & (compustat_lpermno['datadate'] <= compustat_lpermno['LINKENDDT'])
    )
].copy()

compustat_lpermno = compustat_lpermno.sort_values(['gvkey', 'datadate', 'LINKDT', 'LINKENDDT'])
compustat_lpermno = compustat_lpermno.drop_duplicates(subset=['gvkey', 'datadate'], keep='first')
compustat = compustat_lpermno.drop(columns=['LINKDT', 'LINKENDDT'])
compustat['LPERMNO'] = pd.to_numeric(compustat['LPERMNO'], errors='coerce').astype('Int64')

matched_lpermno_pct = round(compustat['LPERMNO'].notna().mean() * 100, 2)
print('Matched LPERMNO percentage:', matched_lpermno_pct)
compustat[['gvkey', 'datadate', 'LPERMNO']].head()

Matched LPERMNO percentage: 49.16


,gvkey,datadate,LPERMNO
0,001004,2019-02-28,54594
1,001004,2019-05-31,54594
2,001004,2019-08-31,54594
3,001004,2019-11-30,54594
4,001004,2020-02-29,54594


## 9. Create Financial Ratios

In [7]:
compustat['roa'] = compustat['niq'] / compustat['atq']
compustat['asset_turnover'] = compustat['saleq'] / compustat['atq']

compustat[['gvkey', 'LPERMNO', 'datadate', 'rdq', 'roa', 'asset_turnover']].head()

,gvkey,LPERMNO,datadate,rdq,roa,asset_turnover
0,001004,54594,2019-02-28,2019-03-19,-0.024184,0.342386
1,001004,54594,2019-05-31,2019-07-10,0.015028,0.370881
2,001004,54594,2019-08-31,2019-09-25,0.002614,0.321728
3,001004,54594,2019-11-30,2019-12-19,0.008092,0.319619
4,001004,54594,2020-02-29,2020-03-24,0.001258,0.307781


## 10. Save Cleaned Compustat Output

In [8]:
compustat.to_csv(OUTPUT_PATH, index=False)

print('Saved cleaned dataset to:', OUTPUT_PATH)
print('Saved missing-value summary to:', MISSING_PATH)
print('Final cleaned Compustat shape:', compustat.shape)

Saved cleaned dataset to: compustat_cleaned.csv
Saved missing-value summary to: compustat_missing_pct.csv
Final cleaned Compustat shape: (317191, 28)


## 11. Load And Prepare Monthly CRSP Data

In [9]:
crsp_monthly = pd.read_csv(CRSP_MONTHLY_PATH)
crsp_monthly['date'] = pd.to_datetime(crsp_monthly['date'], errors='coerce')
crsp_monthly['PERMNO'] = pd.to_numeric(crsp_monthly['PERMNO'], errors='coerce').astype('Int64')

print('Loaded CRSP monthly data:', crsp_monthly.shape)
crsp_monthly.head()

Loaded CRSP monthly data: (322716, 8)


,PERMNO,ret_m,vol_d,n_days,prc_end,mv_end,vol_avg,date
0,10026,-0.100016,0.025857,21,165.84,3137527.0,106825.71,2020-01-01
1,10026,-0.030271,0.012975,19,160.82,3042553.8,98146.69,2020-02-01
2,10026,-0.243679,0.079814,22,121.00,2285448.0,178647.60,2020-03-01
3,10026,0.049833,0.046392,21,127.03,2399342.5,169858.81,2020-04-01
4,10026,0.012596,0.039382,20,128.63,2429563.5,137668.90,2020-05-01


## 12. Merge Compustat With CRSP Monthly

Merge rule:
- each quarter becomes active in the month of its `rdq`
- monthly CRSP observations use the most recent quarter whose `rdq_month` is less than or equal to that stock month
- stock linking uses `LPERMNO = PERMNO`

In [10]:
compustat_for_merge = compustat.copy()
compustat_for_merge['rdq_month'] = compustat_for_merge['rdq'].dt.to_period('M').dt.to_timestamp()
compustat_for_merge = compustat_for_merge.sort_values(['rdq_month', 'LPERMNO']).reset_index(drop=True)

crsp_monthly = crsp_monthly.sort_values(['date', 'PERMNO']).reset_index(drop=True)

merged = pd.merge_asof(
    crsp_monthly,
    compustat_for_merge,
    left_on='date',
    right_on='rdq_month',
    left_by='PERMNO',
    right_by='LPERMNO',
    direction='backward',
    allow_exact_matches=True,
)

merge_match_pct = round(merged['gvkey'].notna().mean() * 100, 2)
print('Merged dataset shape:', merged.shape)
print('CRSP rows matched to Compustat features (%):', merge_match_pct)
merged[['PERMNO', 'date', 'gvkey', 'datadate', 'rdq', 'roa', 'asset_turnover']].head(10)

Merged dataset shape: (322716, 37)
CRSP rows matched to Compustat features (%): 78.48


,PERMNO,date,gvkey,datadate,rdq,roa,asset_turnover
0,10026,2020-01-01,012825,2019-12-31,2020-01-27,0.015563,0.258093
1,10032,2020-01-01,012945,2019-12-31,2020-01-22,0.014694,0.403965
2,10044,2020-01-01,011976,2019-11-30,2020-01-13,-0.002468,0.271232
3,10051,2020-01-01,016456,2019-09-30,2019-11-07,0.007101,0.348932
4,10065,2020-01-01,001119,2019-12-31,2020-01-30,NaN,NaN
5,10104,2020-01-01,012142,2019-11-30,2019-12-12,0.023476,0.097661
6,10107,2020-01-01,012141,2019-12-31,2020-01-29,0.041193,0.130505
7,10138,2020-01-01,012138,2019-12-31,2020-01-29,0.058443,0.157410
8,10145,2020-01-01,001300,2019-12-31,2020-01-31,0.026619,0.161830
9,10158,2020-01-01,185128,2019-09-30,2019-11-05,0.007579,0.181164


## 13. Keep Only Successfully Matched Rows

In [11]:
merged_matched_only = merged[merged['gvkey'].notna()].copy()

print('Matched-only dataset shape:', merged_matched_only.shape)
merged_matched_only[['PERMNO', 'date', 'gvkey', 'datadate', 'rdq', 'roa', 'asset_turnover']].head(10)

Matched-only dataset shape: (253281, 37)


,PERMNO,date,gvkey,datadate,rdq,roa,asset_turnover
0,10026,2020-01-01,012825,2019-12-31,2020-01-27,0.015563,0.258093
1,10032,2020-01-01,012945,2019-12-31,2020-01-22,0.014694,0.403965
2,10044,2020-01-01,011976,2019-11-30,2020-01-13,-0.002468,0.271232
3,10051,2020-01-01,016456,2019-09-30,2019-11-07,0.007101,0.348932
4,10065,2020-01-01,001119,2019-12-31,2020-01-30,NaN,NaN
5,10104,2020-01-01,012142,2019-11-30,2019-12-12,0.023476,0.097661
6,10107,2020-01-01,012141,2019-12-31,2020-01-29,0.041193,0.130505
7,10138,2020-01-01,012138,2019-12-31,2020-01-29,0.058443,0.157410
8,10145,2020-01-01,001300,2019-12-31,2020-01-31,0.026619,0.161830
9,10158,2020-01-01,185128,2019-09-30,2019-11-05,0.007579,0.181164


## 14. Missing Percentage In Matched-Only Dataset

In [12]:
matched_only_missing_pct = (
    (merged_matched_only.isna().mean() * 100)
    .round(2)
    .rename('missing_pct')
    .reset_index()
    .rename(columns={'index': 'column'})
    .sort_values('missing_pct', ascending=False)
)

matched_only_missing_pct.to_csv(MATCHED_ONLY_MISSING_PATH, index=False)
matched_only_missing_pct

,column,missing_pct
26,dlcq,14.22
27,dlttq,6.03
35,asset_turnover,5.40
34,roa,5.38
23,ceqq,5.38
28,ltq,5.30
22,atq,5.30
24,cheq,5.30
25,cogsq,5.02
30,saleq,4.91


## 15. Save Full And Matched-Only Merged Datasets

In [13]:
merged.to_csv(MERGED_OUTPUT_PATH, index=False)
merged_matched_only.to_csv(MATCHED_ONLY_OUTPUT_PATH, index=False)

print('Saved full merged dataset to:', MERGED_OUTPUT_PATH)
print('Saved matched-only dataset to:', MATCHED_ONLY_OUTPUT_PATH)
print('Saved matched-only missing summary to:', MATCHED_ONLY_MISSING_PATH)
print('Final full merged shape:', merged.shape)
print('Final matched-only shape:', merged_matched_only.shape)

Saved full merged dataset to: compustat_crsp_merged.csv
Saved matched-only dataset to: compustat_crsp_merged_matched_only.csv
Saved matched-only missing summary to: compustat_crsp_merged_matched_only_missing_pct.csv
Final full merged shape: (322716, 37)
Final matched-only shape: (253281, 37)
